In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import Adam
from torchvision.transforms.functional import to_tensor
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

X_test_t  = torch.tensor(X_test, dtype=torch.float32)
y_test_t  = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

In [ ]:
from torch.utils.data import TensorDataset
# 2. Create TensorDataset objects
train_ds = TensorDataset(X_train_t, y_train_t)
test_ds  = TensorDataset(X_test_t, y_test_t)

In [ ]:
# 3. Create DataLoaders

# DataLoader for training data
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False)

In [ ]:
# 4. Print shape of one batch
sample_batch_images, sample_batch_labels = next(iter(train_loader))

print(f"Batch image shape: {sample_batch_images.shape}")
print(f"Batch label shape: {sample_batch_labels.shape}")
print(f"First few labels in batch: {sample_batch_labels[:5]}")

In [ ]:
# 5. Display sample images

In [ ]:
# Task 1: Write your model class here:
class NN4Layer(nn.Module):

    def __init__(self, input_dim, hidden_dim, output_dim):
        super(NN4Layer, self).__init__()

        # First linear layer: input features -> hidden layer
        self.layer1 = nn.Linear(input_dim, hidden_dim)

        # Second linear layer: hidden layer -> hidden layer
        self.layer2 = nn.Linear(hidden_dim, hidden_dim)

        self.layer3 = nn.Linear(hidden_dim, hidden_dim)

        self.layer4 = nn.Linear(hidden_dim, 1)

        # ReLU activation for non-linearity
        self.relu = nn.ReLU()

    # Defines how input data flows through the network
    def forward(self, x):
        # First hidden layer
        z1 = self.layer1(x)
        a1 = self.relu(z1)

        # Second hidden layer
        z2 = self.layer2(a1)
        a2 = self.relu(z2)


        z3 = self.layer2(a2)
        a3 = self.relu(z3)
        # Output layer (raw scores / logits)
        output = self.layer3(a3)

        return output


In [ ]:
# Task 2: Write your training loop here:

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0

    for Xb, yb in loader:
        Xb, yb = Xb.to(device), yb.to(device)

        optimizer.zero_grad()
        pred = model(Xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * Xb.size(0)

    return running_loss / len(loader.dataset)

In [ ]:
# Task 3: Write your validation loop here:
@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0

    for Xb, yb in loader:
        Xb, yb = Xb.to(device), yb.to(device)
        pred = model(Xb)
        loss = criterion(pred, yb)
        running_loss += loss.item() * Xb.size(0)

    return running_loss / len(loader.dataset)

In [ ]:
# Task 4: Define device, model, loss, optimizer:
num_classes = int(np.unique(y_train).size)

model = NN4Layer(input_dim=X_train_t.shape[1], hidden_dim=20, output_dim=num_classes).to(device)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

def accuracy_from_logits(logits, y_true):
    preds = logits.argmax(dim=1)
    return (preds == y_true).float().mean().item()

In [ ]:
# Task 5: Start training for 20 epochs:input_dim = X_train_t.shape[1]

epochs = 20
train_losses, val_losses = [], []

for epoch in range(epochs):
    tr = train_one_epoch(model, train_loader, criterion, optimizer, device)
    va = validate(model, test_loader, criterion, device)
    train_losses.append(tr); val_losses.append(va)

    print(f"Epoch {epoch+1}/{epochs} | train={tr:.4f} | val={va:.4f}")

In [ ]:
# Task 1: Write your code here:
model.eval()
with torch.no_grad():
    preds = model(X_test_t.to(device)).cpu().numpy().ravel()

rmse = mean_squared_error(y_test.values, preds, squared=False)
mae  = mean_absolute_error(y_test.values, preds)
r2   = r2_score(y_test.values, preds)
print("RMSE:", rmse, "MAE:", mae, "R2:", r2)

In [ ]:
# Task 2 (Bonus): Write your code here: